In [53]:
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from sklearn.datasets import fetch_20newsgroups
import json
import pickle
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import defaultdict

### Data Preparation

In [3]:
data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

In [30]:
documents = data.data[:2000]

print("Number of documents: ", len(documents))
print("Document samples: ", documents[:5])


Number of documents:  2000
Document samples:  ['I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a Bricklin. The doors were really small. In addition,\nthe front bumper was separate from the rest of the body. This is \nall I know. If anyone can tellme a model name, engine specs, years\nof production, where this car is made, history, or whatever info you\nhave on this funky looking car, please e-mail.', "A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if y

In [17]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing(document):
    document = document.lower()
    document = re.sub(r"\S+@\S+", " ", document)
    document = re.sub(r"[^a-zA-Z0-9\s]", " ", document)
    tokens = document.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [t for t in tokens if len(t) >= 3]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens

[nltk_data] Downloading package stopwords to C:\Users\MSI
[nltk_data]     GF66\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\MSI
[nltk_data]     GF66\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [18]:
preprocessed_documents = [preprocessing(doc) for doc in documents]

In [25]:
print(preprocessed_documents[:5])
print(len(preprocessed_documents[0]))

[['wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sport', 'car', 'looked', 'late', '60', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please', 'mail'], ['fair', 'number', 'brave', 'soul', 'upgraded', 'clock', 'oscillator', 'shared', 'experience', 'poll', 'please', 'send', 'brief', 'message', 'detailing', 'experience', 'procedure', 'top', 'speed', 'attained', 'cpu', 'rated', 'speed', 'add', 'card', 'adapter', 'heat', 'sink', 'hour', 'usage', 'per', 'day', 'floppy', 'disk', 'functionality', '800', 'floppy', 'especially', 'requested', 'summarizing', 'next', 'two', 'day', 'please', 'add', 'network', 'knowledge', 'base', 'done', 'clock', 'upgrade', 'answered', 'poll', 'thanks'], ['well', 'folk', 'mac', 'plus', 'finally', 'gave', 'g

In [61]:
document_frequency = defaultdict(int)

for doc in preprocessed_documents:
    unique_tokens = set(doc)
    for token in unique_tokens:
        document_frequency[token] += 1


num_documents = len(preprocessed_documents)
filtered_documents = []

for doc in preprocessed_documents:
    new_doc = [token for token in doc if document_frequency[token] >= 10 and document_frequency[token] <= (0.7 * num_documents)]
    filtered_documents.append(new_doc)

In [62]:
print(filtered_documents[:5])

[['wondering', 'anyone', 'could', 'car', 'saw', 'day', 'door', 'sport', 'car', 'looked', 'late', 'early', 'called', 'door', 'really', 'small', 'addition', 'front', 'separate', 'rest', 'body', 'know', 'anyone', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'looking', 'car', 'please', 'mail'], ['fair', 'number', 'brave', 'soul', 'clock', 'shared', 'experience', 'please', 'send', 'brief', 'message', 'experience', 'procedure', 'top', 'speed', 'cpu', 'speed', 'add', 'card', 'adapter', 'heat', 'hour', 'per', 'day', 'floppy', 'disk', '800', 'floppy', 'especially', 'requested', 'next', 'two', 'day', 'please', 'add', 'network', 'knowledge', 'base', 'done', 'clock', 'upgrade', 'answered', 'thanks'], ['well', 'folk', 'mac', 'plus', 'finally', 'gave', 'starting', 'life', 'way', 'back', '1985', 'market', 'new', 'machine', 'bit', 'intended', 'looking', '160', 'maybe', 'bunch', 'question', 'hopefully', 'somebody', 'answer', 'anybody', 'know', '

### Model Training

In [63]:
dictionary = Dictionary(filtered_documents)

corpus = [dictionary.doc2bow(doc) for doc in filtered_documents]

In [65]:
lda_model = LdaModel(corpus=corpus,
                     id2word=dictionary,
                     num_topics=15,
                     passes=50,
                     iterations=200,
                     alpha='auto',
                     eta='auto',
                     random_state=42)

In [67]:
with open("lda_model.pkl", "wb") as file:
    pickle.dump(lda_model, file)

#### Topic Labeling

In [83]:
with open("lda_model.pkl", "rb") as file:
    lda_model = pickle.load(file)

dictionary = lda_model.id2word


num_topics = lda_model.num_topics

for topic_id, topic in lda_model.show_topics(num_topics=num_topics, num_words=20, formatted=True):
    print(f"Topic {topic_id}: {topic}")

Topic 0: 0.020*"available" + 0.020*"version" + 0.014*"ftp" + 0.013*"edu" + 0.012*"motif" + 0.012*"system" + 0.012*"server" + 0.011*"machine" + 0.011*"contact" + 0.010*"software" + 0.010*"widget" + 0.010*"window" + 0.009*"sun" + 0.009*"source" + 0.008*"type" + 0.008*"mit" + 0.008*"pub" + 0.008*"file" + 0.008*"export" + 0.008*"also"
Topic 1: 0.022*"god" + 0.021*"jesus" + 0.018*"christian" + 0.014*"believe" + 0.014*"say" + 0.012*"one" + 0.012*"think" + 0.011*"mean" + 0.011*"word" + 0.011*"would" + 0.010*"true" + 0.010*"people" + 0.010*"truth" + 0.009*"may" + 0.009*"belief" + 0.009*"bible" + 0.008*"make" + 0.008*"church" + 0.008*"faith" + 0.007*"know"
Topic 2: 0.022*"good" + 0.011*"sin" + 0.009*"year" + 0.009*"excellent" + 0.009*"father" + 0.008*"one" + 0.008*"way" + 0.007*"spirit" + 0.007*"son" + 0.007*"would" + 0.007*"god" + 0.006*"get" + 0.006*"question" + 0.006*"israel" + 0.005*"missing" + 0.005*"cover" + 0.005*"condition" + 0.005*"day" + 0.005*"offer" + 0.005*"car"
Topic 3: 0.014*"spa

In [103]:
topic_labels = {
    "0": "Software Distribution & System Tools",
    "1": "Christian Religion & Belief",
    "2": "Morality, Faith, Philosophy",
    "3": "Space Explorations",
    "4": "Science, Technology & Academia",
    "5": "Computer Hardware Troubleshooting",
    "6": "Help Requests & General Q/A",
    "7": "Programming and Code",
    "8": "Logic, Arguments & Reasoning",
    "9": "Politics, Government & Rights",
    "10": "Armenian Conflict & Middle East",
    "11": "Sports, Games & Teams",
    "12": "Health & Diseases",
    "13": "General Opinions, Questions & Advice",
    "14": "Gun Control, Encryption & Law"
}

with open("topic_labels.json", "w") as f:
    json.dump(topic_labels, f, indent=4)

for topic_id, topic in topic_labels.items():
    print(f"Topic {topic_id}: {topic}")

Topic 0: Software Distribution & System Tools
Topic 1: Christian Religion & Belief
Topic 2: Morality, Faith, Philosophy
Topic 3: Space Explorations
Topic 4: Science, Technology & Academia
Topic 5: Computer Hardware Troubleshooting
Topic 6: Help Requests & General Q/A
Topic 7: Programming and Code
Topic 8: Logic, Arguments & Reasoning
Topic 9: Politics, Government & Rights
Topic 10: Armenian Conflict & Middle East
Topic 11: Sports, Games & Teams
Topic 12: Health & Diseases
Topic 13: General Opinions, Questions & Advice
Topic 14: Gun Control, Encryption & Law


### Inference

In [104]:
def classify(text):
    tokens = preprocessing(text)
    bow = dictionary.doc2bow(tokens)

    topic_distribution = lda_model.get_document_topics(bow)
    topic_distribution = sorted(topic_distribution, key=lambda x: x[1], reverse=True)
    top3 = topic_distribution[:3]

    for topic_id, prob in top3:
        name = topic_labels[str(topic_id)]
        print(f"Topic: {name}")
        print(f"Probability: {prob:.4f}")

    return top3

In [105]:
samples = [
    "The new graphics card delivers amazing performance for gaming. The GPU can handle 4K resolution easily with ray tracing enabled. Gamers will love the improved frame rates.",

    "Scientists discovered a new exoplanet orbiting a distant star in the habitable zone. The research team published their findings in Nature journal. This discovery could provide insights into planetary formation.",

    "The basketball team won the championship after an incredible final game. The players celebrated with fans in the stadium. It was the team's first title in twenty years.",

    "Congress passed a new bill regarding healthcare reform. The president is expected to sign the legislation next week. The policy will affect millions of citizens across the country.",

    "I love cooking Italian food at home. Pasta carbonara and margherita pizza are my favorite dishes to make. Fresh ingredients make all the difference in authentic recipes."
]

In [106]:
for i, doc in enumerate(samples, start=1):
    print(f"\nTop 3 topics for Sample {i}")
    classify(doc)


Top 3 topics for Sample 1
Topic: Computer Hardware Troubleshooting
Probability: 0.5766
Topic: Morality, Faith, Philosophy
Probability: 0.2349
Topic: Space Explorations
Probability: 0.1467

Top 3 topics for Sample 2
Topic: Science, Technology & Academia
Probability: 0.5483
Topic: Sports, Games & Teams
Probability: 0.2794
Topic: Christian Religion & Belief
Probability: 0.1272

Top 3 topics for Sample 3
Topic: Sports, Games & Teams
Probability: 0.8259
Topic: Programming and Code
Probability: 0.1158

Top 3 topics for Sample 4
Topic: General Opinions, Questions & Advice
Probability: 0.3886
Topic: Armenian Conflict & Middle East
Probability: 0.3338
Topic: Politics, Government & Rights
Probability: 0.2469

Top 3 topics for Sample 5
Topic: Health & Diseases
Probability: 0.3734
Topic: Armenian Conflict & Middle East
Probability: 0.3579
Topic: Christian Religion & Belief
Probability: 0.2047
